# Pix2Pix — notebook de trabajo en Colab

Restauración de fotografías · Redes Neuronales y Deep Learning (UAO).

Une los dos notebooks que usó el equipo:

- del de **base + ablación** toma la estructura, `RESUME` y la comparación con la ablación;
- del de **métricas** (el que sí corrió) toma guardar todo en Drive y desempaquetar el dataset en `/content`;
- agrega la **evaluación del split de prueba**, la tabla automática, las figuras y la demo.

La corrida que respalda las cifras del informe está en `pix2pix_entrenamiento_colab.ipynb` (evidencia, no se edita).

| Sección | Qué hace | ¿Obligatoria? |
|---|---|---|
| 0 | Configuración: rutas y qué correr | Sí |
| 1 | Drive, repositorio, dependencias y dataset | Sí |
| 2 | Entrenar la corrida base | Solo si no existe el checkpoint |
| 3 | Entrenar la ablación sin GAN | Opcional |
| 4 | Curvas de entrenamiento | Sí |
| 5 | Evaluación del split `test` + tabla + figuras | Sí |
| 6 | Demo (Gradio, enlace temporal) | Para la captura |
| 7 | Descargar un zip ligero | Sí |

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU*.

## 0. Configuración

In [ ]:
from pathlib import Path

# Carpeta del proyecto en Drive: aquí quedan checkpoints, métricas y muestras.
DRIVE_ROOT = Path('/content/drive/MyDrive/Redes-Pix2Pix')
OUT_ROOT = DRIVE_ROOT / 'resultados_pix2pix'          # base/ y ablation/

# Dónde buscar el zip del dataset (rutas que usó el equipo, en orden).
# La carpeta del equipo (https://drive.google.com/drive/folders/1SADEgsZbRjcOhBGSgQsxNNcRtu3O-JIP)
# aparece en "Compartido conmigo": en Drive, clic derecho → Organizar → Agregar acceso directo
# → "Mi unidad". Si no está en estas rutas, la celda 1 lo busca en toda "Mi unidad".
ZIP_CANDIDATOS = [
    DRIVE_ROOT / 'datasetv1.zip',
    Path('/content/drive/MyDrive/datasetv1.zip'),
    Path('/content/datasetv1.zip'),
    Path('/content/datasetv1(1).zip'),
]

# Checkpoint de la corrida que ya existe (20 épocas, ejecutada por Isabela). Si está, no hace
# falta reentrenar. Está en el Drive de Isabela: si te comparte la carpeta
# `metricas_loss_existente`, agrega un acceso directo en "Mi unidad" y se encuentra sola.
CKPT_CANDIDATOS = [
    # Corrida de Cristian (45 épocas), subida a la carpeta compartida Redes-Pix2Pix/checkpoints/
    DRIVE_ROOT / 'checkpoints' / 'base' / 'checkpoints',
    Path('/content/drive/MyDrive/pix2pix_outputs/metricas_loss_existente/checkpoints'),
    Path('/content/drive/MyDrive/metricas_loss_existente/checkpoints'),
]
CKPT_EXISTENTE = next((p for p in CKPT_CANDIDATOS if p.is_dir()), CKPT_CANDIDATOS[0])

ENTRENAR_BASE = False      # True para entrenar la base desde este notebook
EPOCAS_BASE = 100          # base.yaml usa 100 con early stopping; la corrida existente usó 20
ENTRENAR_ABLACION = False  # True para entrenar la U-Net solo L1
RESUME = False             # True para continuar tras una desconexión de Colab

REPO_URL = 'https://github.com/Isabela1105/Trabajo-Final-de-Redes.git'
REPO = Path('/content/Trabajo-Final-de-Redes')

## 1. Drive, repositorio, dependencias y dataset

In [ ]:
import os, re, subprocess, sys, zipfile
from google.colab import drive

drive.mount('/content/drive')
OUT_ROOT.mkdir(parents=True, exist_ok=True)
# Se recalcula después de montar Drive (en la celda 0 Drive aún no estaba montado).
CKPT_EXISTENTE = next((p for p in CKPT_CANDIDATOS if p.is_dir()), None) or next(
    (p for p in Path('/content/drive/MyDrive').glob('**/metricas_loss_existente/checkpoints')),
    CKPT_CANDIDATOS[0])
print('Checkpoint existente:', CKPT_EXISTENTE, '(encontrado)' if CKPT_EXISTENTE.is_dir() else '(no está)')

if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '-q', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
print('Commit del repo:', subprocess.check_output(['git', 'log', '-1', '--format=%h %s'], text=True).strip())

# Colab ya trae TensorFlow con GPU: se instala el resto sin versiones rígidas.
paquetes = []
for linea in Path('requirements.txt').read_text(encoding='utf-8').splitlines():
    linea = linea.split('#')[0].strip()
    if linea:
        nombre = re.split(r'[<>=!~\[]', linea)[0].strip()
        if not nombre.lower().startswith('tensorflow'):
            paquetes.append(nombre)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *paquetes], check=True)

import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
ZIP = next((p for p in ZIP_CANDIDATOS if p.is_file()), None)
if ZIP is None:  # acceso directo a la carpeta del equipo u otra ubicación en "Mi unidad"
    ZIP = next(Path('/content/drive/MyDrive').glob('**/datasetv1.zip'), None)
assert ZIP is not None, 'Sube datasetv1.zip a una de estas rutas: ' + ', '.join(map(str, ZIP_CANDIDATOS))

DATA = REPO / 'data' / 'processed'
with zipfile.ZipFile(ZIP) as z:
    tops = {n.split('/')[0] for n in z.namelist() if n.strip() and not n.startswith('__MACOSX')}
    destino = REPO if 'data' in tops else REPO / 'data' if 'processed' in tops else DATA
    z.extractall(destino)
print('Dataset:', ZIP, '→', destino)

from src.gan.data.tf_dataset import list_pairs
for split in ('train', 'val', 'test'):
    entradas, objetivos = list_pairs(DATA / split)
    print(f'{split:5s}: {len(entradas)} pares')
    assert entradas, f'No hay pares en {DATA / split}'

## 2. Entrenar la corrida base

`outputs/base` del repo apunta (enlace simbólico) a `OUT_ROOT/base` en Drive: si Colab se desconecta, pon `RESUME = True` y vuelve a ejecutar.

In [ ]:
import yaml

def preparar_salida(nombre):
    salida = OUT_ROOT / nombre
    salida.mkdir(parents=True, exist_ok=True)
    enlace = REPO / 'outputs' / nombre
    enlace.parent.mkdir(exist_ok=True)
    if enlace.is_symlink():
        enlace.unlink()
    elif enlace.exists():
        raise RuntimeError(f'{enlace} existe y no es un enlace; revísalo antes de seguir')
    enlace.symlink_to(salida, target_is_directory=True)
    return salida

def entrenar(config_path, resume):
    cmd = [sys.executable, '-m', 'src.gan.train', '--config', str(config_path)]
    if resume:
        cmd.append('--resume')
    subprocess.run(cmd, check=True)

if ENTRENAR_BASE:
    BASE_OUT = preparar_salida('base')
    cfg = yaml.safe_load(Path('configs/base.yaml').read_text(encoding='utf-8'))
    cfg['train']['epochs'] = EPOCAS_BASE
    cfg['output']['dir'] = str(BASE_OUT)
    base_cfg = REPO / 'configs' / 'colab_base.yaml'
    base_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
    entrenar(base_cfg, RESUME)
else:
    print('Se omite: ENTRENAR_BASE = False')

## 3. Ablación: U-Net solo L1 (sin discriminador)

In [ ]:
if ENTRENAR_ABLACION:
    ABL_OUT = preparar_salida('ablation')
    cfg = yaml.safe_load(Path('configs/base.yaml').read_text(encoding='utf-8'))
    abl = yaml.safe_load(Path('configs/ablation_l1.yaml').read_text(encoding='utf-8'))
    # Igual que la base; solo cambian use_gan y lambda_l1.
    cfg['train']['use_gan'] = abl['train']['use_gan']        # False
    cfg['train']['lambda_l1'] = abl['train']['lambda_l1']    # 1000.0
    cfg['train']['epochs'] = EPOCAS_BASE
    cfg['output']['dir'] = str(ABL_OUT)
    abl_cfg = REPO / 'configs' / 'colab_ablation.yaml'
    abl_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
    entrenar(abl_cfg, RESUME)
else:
    print('Se omite: ENTRENAR_ABLACION = False')

## 4. Curvas de entrenamiento

In [ ]:
from IPython.display import Image, display

CORRIDAS = {
    'base': OUT_ROOT / 'base' if (OUT_ROOT / 'base' / 'metrics.csv').is_file() else CKPT_EXISTENTE.parent,
    'ablation': OUT_ROOT / 'ablation' if (OUT_ROOT / 'ablation' / 'metrics.csv').is_file()
                else DRIVE_ROOT / 'checkpoints' / 'ablation',
}
for nombre, carpeta in CORRIDAS.items():
    metrics = carpeta / 'metrics.csv'
    if not metrics.is_file():
        print(f'{nombre}: sin metrics.csv en {carpeta}')
        continue
    png = OUT_ROOT / f'curvas_{nombre}.png'
    subprocess.run([sys.executable, 'scripts/plot_training_log.py', str(metrics), str(png),
                    '--title', f'Corrida {nombre} ({carpeta.name})'], check=True)
    display(Image(filename=str(png)))

## 5. Evaluación del split de prueba

Evalúa Entrada, Mediana 3×3, Pix2Pix y (si existe) la ablación sobre las 45 imágenes de `test`, inserta la tabla en README, informe y `docs/RESULTADOS.md`, y copia las figuras a `docs/img/`. Ninguna cifra se escribe a mano.

In [ ]:
import pandas as pd

def checkpoint_de(carpeta):
    return carpeta if carpeta.is_dir() and tf.train.latest_checkpoint(str(carpeta)) else None

CKPT_BASE = checkpoint_de(OUT_ROOT / 'base' / 'checkpoints') or checkpoint_de(CKPT_EXISTENTE)
CKPT_ABL = (checkpoint_de(OUT_ROOT / 'ablation' / 'checkpoints')
            or checkpoint_de(DRIVE_ROOT / 'checkpoints' / 'ablation' / 'checkpoints'))
assert CKPT_BASE, 'No hay checkpoint de la base: entrena (sección 2) o revisa CKPT_EXISTENTE'
print('Pix2Pix  :', tf.train.latest_checkpoint(str(CKPT_BASE)))
print('Ablación :', CKPT_ABL and tf.train.latest_checkpoint(str(CKPT_ABL)))

EVAL = REPO / 'outputs' / 'eval'
cmd = [sys.executable, '-m', 'src.gan.evaluate', '--config', 'configs/base.yaml',
       '--checkpoint', str(CKPT_BASE), '--out', str(EVAL)]
if CKPT_ABL:
    cmd += ['--model', f'U-Net solo L1={CKPT_ABL}']
subprocess.run(cmd, check=True)

display(pd.read_csv(EVAL / 'summary.csv'))
display(pd.read_csv(EVAL / 'by_degradation.csv'))

In [ ]:
subprocess.run([sys.executable, 'scripts/results_table.py', '--summary', str(EVAL / 'summary.csv'),
                '--source', 'outputs/eval/summary.csv',
                '--update', 'README.md', 'report/informe.md', 'docs/RESULTADOS.md'], check=True)
import shutil
for figura in ('results_grid.png', 'failures.png'):
    shutil.copy(EVAL / figura, REPO / 'docs' / 'img' / figura)
    display(Image(filename=str(REPO / 'docs' / 'img' / figura)))
print(subprocess.check_output(['git', 'status', '--short'], text=True))

## 6. Demo

Gradio crea un enlace público temporal; úsalo para tomar la captura `docs/img/app.png`. La demo principal de la defensa es local y sin internet: `python -m app.server --checkpoint outputs/base/checkpoints`. Detén esta celda (■) para seguir.

In [ ]:
!python -m app.gradio_app --checkpoint "{CKPT_BASE}"

## 7. Descargar un zip ligero (sin checkpoints)

In [ ]:
from google.colab import files

PAQUETE = Path('/content/entrega_pix2pix')
if PAQUETE.exists():
    shutil.rmtree(PAQUETE)
for archivo in ['README.md', 'report/informe.md', 'docs/RESULTADOS.md',
                'docs/img/results_grid.png', 'docs/img/failures.png']:
    destino = PAQUETE / 'repo' / archivo
    destino.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(REPO / archivo, destino)
shutil.copytree(EVAL, PAQUETE / 'eval')
for nombre in ('base', 'ablation'):
    if (OUT_ROOT / nombre).is_dir():
        shutil.copytree(OUT_ROOT / nombre, PAQUETE / 'corridas' / nombre,
                        ignore=shutil.ignore_patterns('checkpoints', 'ckpt-*'))
for png in OUT_ROOT.glob('curvas_*.png'):
    shutil.copy(png, PAQUETE / png.name)
files.download(shutil.make_archive('/content/entrega_pix2pix', 'zip', PAQUETE))
print('La carpeta repo/ del zip tiene los archivos para subir al repositorio.')